
> ### Keyword Coverage Analysis
> 
> 
> This notebook provides a diagnostic audit of our `matched_titlewords` across the universal postings dataset. The objective is to identify which keywords are **mandatory** (sole matches driving data inclusion) versus **optional** (supplementary metadata found alongside other matches).
> This analysis helps determine where the current keyword list is overly broad / redundant. By isolating the "sole-match" keywords, we can map the minimal set of terms required to maintain our current capture rate.

> **Core goals:**
> * Quantify the "must-have" dictionary (mandatory keywords).
> * Identify noise or redundant keywords in multi-match postings.
> * Evaluate the impact of removing specific keywords on overall data volume.

### Load (construct) universal postings history

In [ ]:
from jobscraping.processing.postings_utils import unify_postings
from jobscraping.io.files_io import save_data

files = [
    "../data/save/postings/tech/postings_history.json",
    "../data/save/postings/legal/postings_history.json",
    "../data/save/postings/tn/postings_history.json",
    "../data/save/postings/dani/postings_history.json",
]

universal = unify_postings(postings=files, folder_path="")

save_data(
    universal,
    path="../data/save/postings/universal/",
    name="postings_history.json",
    with_timestamp=False,
)

universal_positive = {key: val for key, val in universal.items() if val.get("points") > 0}
save_data(
    universal_positive,
    path="../data/save/postings/universal/",
    name="positive_postings_history.json",
    with_timestamp=False,
)

In [ ]:
matched_titles = [val['matched_titlewords']
                  for key, val in universal_positive.items() # or universal.items() if you want to include all postings
                  if val.get('matched_titlewords')]

# 2. Identify "mandatory" titles 
# A title is mandatory if it appears as the ONLY match for at least one posting.
# We look for postings where the list length is 1, and grab that single element.
mandatory_titles = {
    titles[0] 
    for titles in matched_titles 
    if len(titles) == 1
}

# 3. Identify "non-mandatory" titles
# These are all unique titles found across the dataset, excluding the mandatory ones.
all_unique_titles = {
    title 
    for titles in matched_titles 
    for title in titles
}

non_mandatory_titles = all_unique_titles - mandatory_titles

# Quick verification output
print(f"Mandatory count: {len(mandatory_titles)}")
print(f"Non-mandatory count: {len(non_mandatory_titles)}")

Mandatory count: 69
Non-mandatory count: 45


### Findings

Some mandatory/core titlewords (sole matches):
- account coordinator (414 sole matches)
- account executive (912 sole matches)
- ai developer (5 sole matches)
- ai engineer (21 sole matches)
- anti-money laundering (10 sole matches)
- biotech (2 sole matches)
- business intelligence (75 sole matches)
- business law (2 sole matches)
- capital markets (1 sole matches)
- chemistry informatics (15 sole matches)
- commercial finance (20 sole matches)
- data analyst (18 sole matches)
...

Many titlewords are not mandatory and do not provide unique coverage. These, we can remove for now to reduce script running time.